# 🌍 N-ATLAS Model Server — Kaggle/Colab Deployment

This notebook loads the N-ATLAS LLM and ASR models and serves them as an API.

**Steps:**
1. Install dependencies
2. Login to HuggingFace (gated model)
3. Load N-ATLAS LLM + ASR models
4. Serve via FastAPI + ngrok tunnel

> **GPU Required**: T4 (free on Kaggle/Colab) or better

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers accelerate torch fastapi uvicorn pyngrok librosa soundfile pydantic huggingface_hub

In [ ]:
# Step 2: Login to HuggingFace
# You MUST have accepted access at https://huggingface.co/NCAIR1/N-ATLaS first
from huggingface_hub import login

# Option A: Enter token interactively
login()

# Option B: Paste token directly (uncomment below)
# login(token="hf_YOUR_TOKEN_HERE")

In [ ]:
# Step 3: Load N-ATLAS LLM
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "NCAIR1/N-ATLaS"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading model (this takes 2-5 minutes on T4)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,  # float16 for T4 compatibility
    device_map="auto",
    low_cpu_mem_usage=True
)
print(f"Model loaded on {model.device}")

In [ ]:
# Step 4: Load ASR Models (Whisper)
from transformers import pipeline as hf_pipeline

ASR_MODELS = {
    "ha": "NCAIR1/Hausa-ASR",
    "yo": "NCAIR1/Yoruba-ASR",
    "ig": "NCAIR1/Igbo-ASR",
    "en": "NCAIR1/NigerianAccentedEnglish"
}

asr_pipelines = {}
for lang, model_id in ASR_MODELS.items():
    print(f"Loading ASR: {model_id}...")
    asr_pipelines[lang] = hf_pipeline(
        "automatic-speech-recognition",
        model=model_id,
        chunk_length_s=30,
        device=0
    )
print(f"All {len(asr_pipelines)} ASR models loaded!")

In [ ]:
# Step 5: Test the model
from datetime import datetime

def chat(user_message, language="en", system_prompt=None):
    if system_prompt is None:
        system_prompt = "you are a large language model trained by Awarri AI technologies. You are a friendly assistant and you are here to help."
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
        date_string=datetime.now().strftime("%d %b %Y")
    )
    
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            repetition_penalty=1.12,
            do_sample=True,
            use_cache=True
        )
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

# Test in multiple languages
print("=== English ===")
print(chat("What crops grow best in Northern Nigeria?"))

print("\n=== Hausa ===")
print(chat("Wane irin amfanin gona ne ya fi dacewa a Arewacin Najeriya?"))

print("\n=== Yoruba ===")
print(chat("Kini awọn irugbin ti o dara julọ ni Ariwa Naijiria?"))

In [ ]:
# Step 6: Create FastAPI server
import threading
import uvicorn
from fastapi import FastAPI, UploadFile, File
from pydantic import BaseModel
from typing import List, Optional
import tempfile, os, time, librosa, io, numpy as np

api = FastAPI(title="N-ATLAS API Server")

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: str = "NCAIR1/N-ATLaS"
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.1
    max_tokens: Optional[int] = 512

@api.post("/v1/chat/completions")
async def chat_completions(req: ChatRequest):
    user_msg = req.messages[-1].content
    system_msg = next((m.content for m in req.messages if m.role == 'system'), None)
    response_text = chat(user_msg, system_prompt=system_msg)
    return {
        "id": f"chatcmpl-{int(time.time())}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [{
            "index": 0,
            "message": {"role": "assistant", "content": response_text},
            "finish_reason": "stop"
        }]
    }

@api.post("/v1/audio/transcriptions")
async def transcribe(file: UploadFile = File(...), language: str = "ha"):
    contents = await file.read()
    audio_data, sr = librosa.load(io.BytesIO(contents), sr=16000)
    
    lang = language[:2].lower()
    if lang not in asr_pipelines:
        lang = "en"
    
    result = asr_pipelines[lang](audio_data)
    return {"text": result["text"], "language": lang}

@api.get("/v1/models")
async def list_models():
    return {"data": [{"id": "NCAIR1/N-ATLaS"}, {"id": "NCAIR1/Hausa-ASR"}, {"id": "NCAIR1/Yoruba-ASR"}, {"id": "NCAIR1/Igbo-ASR"}, {"id": "NCAIR1/NigerianAccentedEnglish"}]}

@api.get("/health")
async def health():
    return {"status": "ok", "model": "NCAIR1/N-ATLaS", "gpu": str(model.device)}

# Start server in background thread
def run_server():
    uvicorn.run(api, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("API server started on port 8000")

In [ ]:
# Step 7: Create public tunnel with ngrok
# Sign up at https://ngrok.com and get your auth token (free)

from pyngrok import ngrok

# Set your ngrok auth token
ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE")  # <-- Replace this!

# Create tunnel
public_url = ngrok.connect(8000)
print(f"\n{'='*60}")
print(f"🚀 N-ATLAS API is LIVE at:")
print(f"   {public_url}")
print(f"{'='*60}")
print(f"\nEndpoints:")
print(f"  Chat:       POST {public_url}/v1/chat/completions")
print(f"  Transcribe: POST {public_url}/v1/audio/transcriptions")
print(f"  Models:     GET  {public_url}/v1/models")
print(f"  Health:     GET  {public_url}/health")
print(f"\nUse this URL in your natlas-sdk or agrivoice config!")

In [ ]:
# Step 8: Test the API
import requests

# Test chat
resp = requests.post("http://localhost:8000/v1/chat/completions", json={
    "model": "NCAIR1/N-ATLaS",
    "messages": [{"role": "user", "content": "What is the best way to grow cassava in Nigeria?"}]
})
print(resp.json()["choices"][0]["message"]["content"])

In [ ]:
# Keep the notebook alive — run this cell and leave it running
import time
print("Server is running. Keep this notebook open.")
print(f"API URL: {public_url}")
while True:
    time.sleep(60)
    print(f"[{datetime.now().strftime('%H:%M')}] Server alive. URL: {public_url}")